# 第 1 课：认识 Olist 数据

目标：理解行数、粒度和客户身份；识别直接多表关联的风险。请逐格运行，然后用自己的话回答最后四个问题。

本课只读原始数据；所有分析结论和清洗工作留到后面。

In [3]:
# 1. 导入工具并定位原始数据。Path 能安全表示 Windows 路径。
from pathlib import Path
import pandas as pd

data_dir = Path(r'D:\数据分析\archive')
print('数据目录存在：', data_dir.is_dir())
print('Pandas 版本：', pd.__version__)

数据目录存在： True
Pandas 版本： 2.0.3


## 读取三张表
`dtype=str` 暂时保留 ID 和邮编的原始字符串。`keep_default_na=False, na_values=['']` 仅把空字段当作缺失，保留字面文字。后面会明确转换金额和日期。

In [4]:
orders = pd.read_csv(data_dir / 'olist_orders_dataset.csv', dtype=str, keep_default_na=False, na_values=[''])
customers = pd.read_csv(data_dir / 'olist_customers_dataset.csv', dtype=str, keep_default_na=False, na_values=[''])
items = pd.read_csv(data_dir / 'olist_order_items_dataset.csv', dtype=str, keep_default_na=False, na_values=[''])
orders.head(3)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00


In [5]:
# shape 是 (行数, 列数)。这三张表的一行含义不同。
print('订单：', orders.shape)
print('客户资料：', customers.shape)
print('商品明细：', items.shape)
print('商品明细涉及的订单：', items['order_id'].nunique())

订单： (99441, 8)
客户资料： (99441, 5)
商品明细： (112650, 7)
商品明细涉及的订单： 98666


## 区分两个客户 ID
先观察 `customer_id` 与 `customer_unique_id` 的不同值数，再思考哪一个用于连接订单、哪一个用于识别同一客户。

In [6]:
print('customer_id：', customers['customer_id'].nunique())
print('customer_unique_id：', customers['customer_unique_id'].nunique())

# 找到一位拥有多个 customer_id 的客户，观察其资料。
id_counts = customers['customer_unique_id'].value_counts()
example_customer = id_counts[id_counts > 1].index[0]
customers.loc[customers['customer_unique_id'] == example_customer]

customer_id： 99441
customer_unique_id： 96096


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
14186,1bd3585471932167ab72a84955ebefea,8d50f5eadf50201ccdcedfb9e2ac8455,04045,sao paulo,SP
15321,a8fabc805e9a10a3c93ae5bff642b86b,8d50f5eadf50201ccdcedfb9e2ac8455,04045,sao paulo,SP
16654,897b7f72042714efaa64ac306ba0cafc,8d50f5eadf50201ccdcedfb9e2ac8455,04045,sao paulo,SP
36122,b2b13de0770e06de50080fea77c459e6,8d50f5eadf50201ccdcedfb9e2ac8455,04045,sao paulo,SP
38073,42dbc1ad9d560637c9c4c1533746f86d,8d50f5eadf50201ccdcedfb9e2ac8455,04045,sao paulo,SP
40141,dfb941d6f7b02f57a44c3b7c3fefb44b,8d50f5eadf50201ccdcedfb9e2ac8455,04045,sao paulo,SP
48614,65f9db9dd07a4e79b625effa4c868fcb,8d50f5eadf50201ccdcedfb9e2ac8455,04045,sao paulo,SP
52574,1c62b48fb34ee043310dcb233caabd2e,8d50f5eadf50201ccdcedfb9e2ac8455,04045,sao paulo,SP
58707,a682769c4bc10fc6ef2101337a6c83c9,8d50f5eadf50201ccdcedfb9e2ac8455,04045,sao paulo,SP
67996,6289b75219d757a56c0cce8d9e427900,8d50f5eadf50201ccdcedfb9e2ac8455,04045,sao paulo,SP


In [7]:
# value_counts 统计每个状态有多少行；dropna=False 同时展示空值。
orders['order_status'].value_counts(dropna=False)

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [8]:
# 看空值而不马上填补。未交付订单没有送达时间，可能符合业务含义。
orders.isna().sum().sort_values(ascending=False)

order_delivered_customer_date    2965
order_delivered_carrier_date     1783
order_approved_at                 160
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_estimated_delivery_date       0
dtype: int64

## 用一个教学例子观察关联重复
以下是为了说明机制而造的微型数据，不是 Olist 真实记录。订单 A 有两条商品和两条支付。直接合并后，每个商品行对应两条支付，金额被重复累计。

In [26]:
example_items = pd.DataFrame({'order_id': ['A', 'A'], 'item_id': [1, 2], 'price': [60, 40]})
example_payments = pd.DataFrame({'order_id': ['A', 'A'], 'payment_seq': [1, 2], 'payment_value': [70, 30]})
direct_join = example_items.merge(example_payments, on='order_id')
print('合并前商品金额：', example_items['price'].sum())
print('直接合并后商品金额：', direct_join['price'].sum())
print('合并后行数：', len(direct_join))
direct_join

合并前商品金额： 100
直接合并后商品金额： 200
合并后行数： 4


,order_id,item_id,price,payment_seq,payment_value
0,A,1,60,1,70
1,A,1,60,2,30
2,A,2,40,1,70
3,A,2,40,2,30


In [27]:
# 如果目标是一行一个订单，先分别汇总，再按订单合并。
item_totals = example_items.groupby('order_id', as_index=False)['price'].sum()
payment_totals = example_payments.groupby('order_id', as_index=False)['payment_value'].sum()
order_totals = item_totals.merge(payment_totals, on='order_id', validate='one_to_one')
order_totals

,order_id,price,payment_value
0,A,100,100


In [28]:
# 1. 查看各列的数据类型和非空数量
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


In [29]:
# 2. 统计各列的缺失数量，并从多到少排列
orders.isna().sum().sort_values(ascending=False)

order_delivered_customer_date    2965
order_delivered_carrier_date     1783
order_approved_at                 160
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_estimated_delivery_date       0
dtype: int64

时间处理

In [30]:
# 3. 找到实际送达时间为空的订单
missing_delivery = orders['order_delivered_customer_date'].isna()

# 4. 查看这些订单的状态分布
orders.loc[missing_delivery, 'order_status'].value_counts(dropna=False)

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

In [31]:
# 创建工作副本，保留原始读取结果
orders_clean = orders.copy()

# 标记“已交付，但缺少实际送达时间”的订单
orders_clean['flag_delivered_missing_date'] = (
    orders_clean['order_status'].eq('delivered')
    & orders_clean['order_delivered_customer_date'].isna()
)

print(orders_clean['flag_delivered_missing_date'].sum())

8


In [32]:
# 需要转换的五个时间字段
time_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

# 记录每列有多少条“原本非空，但无法解析”的数据
parse_fail_counts = {}

for col in time_cols:
    orders_clean[col] = pd.to_datetime(
        orders[col],
        format='%Y-%m-%d %H:%M:%S',
        errors='coerce'
    )

    parse_fail_counts[col] = (
        orders[col].notna()
        & orders_clean[col].isna()
    ).sum()

In [33]:
# 日期列现在应该显示为 datetime 类型
print(orders_clean[time_cols].dtypes)

order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


In [34]:
# 只统计“原本有值，转换后却变成缺失”的记录
print(pd.Series(parse_fail_counts, name='解析失败数'))

order_purchase_timestamp         0
order_approved_at                0
order_delivered_carrier_date     0
order_delivered_customer_date    0
order_estimated_delivery_date    0
Name: 解析失败数, dtype: int64


In [35]:
# 给三个时间列起简短的变量名，方便阅读
purchase = orders_clean['order_purchase_timestamp']
carrier = orders_clean['order_delivered_carrier_date']
delivery = orders_clean['order_delivered_customer_date']

# 两端时间都有值，才可以比较
can_check_carrier = purchase.notna() & carrier.notna()
can_check_delivery = carrier.notna() & delivery.notna()

# 标记时间倒置
orders_clean['flag_carrier_before_purchase'] = (
    can_check_carrier & (carrier < purchase)
)

orders_clean['flag_delivery_before_carrier'] = (
    can_check_delivery & (delivery < carrier)
)

In [36]:
time_check_summary = pd.DataFrame({
    '检查项目': ['交物流早于下单', '送达早于交物流'],

    '可比较订单数': [
        can_check_carrier.sum(),
        can_check_delivery.sum()
    ],

    '无法比较订单数': [
        (~can_check_carrier).sum(),
        (~can_check_delivery).sum()
    ],

    '时间倒置订单数': [
        orders_clean['flag_carrier_before_purchase'].sum(),
        orders_clean['flag_delivery_before_carrier'].sum()
    ]
})

time_check_summary

,检查项目,可比较订单数,无法比较订单数,时间倒置订单数
0,交物流早于下单,97658,1783,166
1,送达早于交物流,96475,2966,23


In [37]:
orders_clean.loc[
    orders_clean['flag_delivery_before_carrier'],
    [
        'order_id',
        'order_status',
        'order_purchase_timestamp',
        'order_delivered_carrier_date',
        'order_delivered_customer_date'
    ]
].head(5)

,order_id,order_status,order_purchase_timestamp,order_delivered_carrier_date,order_delivered_customer_date
6437,a1abeb653a4d4cd1e142ccb8c82cd069,delivered,2017-07-20 11:20:52,2017-07-28 16:57:58,2017-07-25 19:32:56
9553,383aa8b2724fe452d9ccd9934a8c628b,delivered,2017-07-02 20:58:43,2017-07-07 17:22:41,2017-07-06 14:27:51
13487,cb1134f9010d242e9515ad1c78ec0c39,delivered,2017-07-16 12:35:34,2017-07-20 19:22:02,2017-07-19 14:13:28
14474,dceb62e8fa94b46006c9554fed743df0,delivered,2017-07-20 20:58:05,2017-08-01 18:23:30,2017-07-26 18:09:10
19268,5f9d46795c3126674e52becb3a1a517f,delivered,2017-07-18 11:48:20,2017-07-20 23:03:42,2017-07-20 18:52:41


In [38]:
#运耗时间transit_days
orders_clean['transit_days_raw'] = (
    orders_clean['order_delivered_customer_date']
    - orders_clean['order_delivered_carrier_date']
).dt.total_seconds() / 86400

valid_transit = (
    orders_clean['order_status'].eq('delivered')
    & can_check_carrier
    & can_check_delivery
    & ~orders_clean['flag_carrier_before_purchase']
    & ~orders_clean['flag_delivery_before_carrier']
)

# 满足条件则保留耗时，否则设为缺失
orders_clean['transit_days'] = (
    orders_clean['transit_days_raw'].where(valid_transit)
)

orders_clean['transit_days'].describe(
    percentiles=[0.5, 0.9]
).round(2)

count    96281.00
mean         9.33
std          8.75
min          0.00
50%          7.10
90%         18.91
max        205.19
Name: transit_days, dtype: float64

在当前已交付状态和时间有效性规则下，96,281 笔订单的平均运输耗时为 9.33 天，中位数为 7.10 天，P90 为 18.91 天。最长耗时为 205.19 天，需进一步核查长耗时记录。

In [39]:
orders_clean['transit_days'].quantile(
    [0.95, 0.99]
).round(2)

0.95    24.22
0.99    40.99
Name: transit_days, dtype: float64

In [40]:
orders_clean.nlargest(5, 'transit_days')[
    [
        'order_id',
        'order_delivered_carrier_date',
        'order_delivered_customer_date',
        'order_estimated_delivery_date',
        'transit_days'
    ]
]

,order_id,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,transit_days
55619,1b3190b2dfa9d789e1f14c05b647a14a,2018-02-26 18:49:07,2018-09-19 23:24:07,2018-03-15,205.190972
19590,ca07593549f1816d26a572e06dc1eab6,2017-03-08 13:47:46,2017-09-19 14:36:39,2017-03-22,195.033947
89130,285ab9426d6982034523a855f55a885e,2017-03-09 08:53:20,2017-09-19 14:00:04,2017-04-06,194.213009
70307,2fb597c2f772eca01b1f5c561bf6cc7b,2017-03-13 08:54:02,2017-09-19 14:33:17,2017-04-17,190.235590
61610,440d0d17af552815d15a9e41abe49359,2017-03-15 13:00:08,2017-09-19 15:12:50,2017-04-07,188.092153


In [41]:
# normalize() 把时间统一到当天 00:00:00，保留日期
actual_day = (
    orders_clean['order_delivered_customer_date'].dt.normalize()
)

estimated_day = (
    orders_clean['order_estimated_delivery_date'].dt.normalize()
)

purchase_day = (
    orders_clean['order_purchase_timestamp'].dt.normalize()
)

# 延迟分析使用自己的有效样本条件
valid_delay = (
    orders_clean['order_status'].eq('delivered')
    & actual_day.notna()
    & estimated_day.notna()
    & (
        orders_clean['order_delivered_customer_date']
        >= orders_clean['order_purchase_timestamp']
    )
    & (estimated_day >= purchase_day)
)

orders_clean['delivery_delay_days'] = (
    (actual_day - estimated_day).dt.days.where(valid_delay)
)

In [42]:
# 只在有效样本中计算，避免把未知结果当成未延迟
delay = orders_clean.loc[valid_delay, 'delivery_delay_days']

print('参与延迟分析的订单数：', len(delay))
print('延迟订单数：', delay.gt(0).sum())
print('延迟率：{:.2%}'.format(delay.gt(0).mean()))

参与延迟分析的订单数： 96470
延迟订单数： 6534
延迟率：6.77%


在 96,470 笔可判断是否延迟的已交付订单中，6,534 笔晚于预计日期送达，延迟率为 6.77%，按期送达率为 93.23%。另有 8 笔已交付订单缺少实际送达时间，未纳入计算。

In [43]:
# 在原始表上检查，避免新增的计算字段影响检查
key_check = pd.Series({
    '总行数': len(orders),
    '订单ID不同值数': orders['order_id'].nunique(),
    '订单ID缺失行数': orders['order_id'].isna().sum(),
    '订单ID重复行数_首条之后': orders['order_id'].duplicated().sum(),
    '整行重复数_首条之后': orders.duplicated().sum()
})

key_check

总行数              99441
订单ID不同值数         99441
订单ID缺失行数             0
订单ID重复行数_首条之后        0
整行重复数_首条之后           0
dtype: int64

订单表共 99,441 行，order_id 非空且唯一，未发现整行重复，符合“一行一个订单”的粒度要求。

In [44]:
status_time_check = (
    orders_clean.assign(
        missing_approved=orders_clean['order_approved_at'].isna(),
        missing_carrier=orders_clean['order_delivered_carrier_date'].isna(),
        missing_delivery=orders_clean['order_delivered_customer_date'].isna()
    )
    .groupby('order_status')
    .agg(
        订单数=('order_id', 'size'),
        审批时间缺失=('missing_approved', 'sum'),
        交物流时间缺失=('missing_carrier', 'sum'),
        送达时间缺失=('missing_delivery', 'sum')
    )
)

status_time_check

,订单数,审批时间缺失,交物流时间缺失,送达时间缺失
order_status,,,,
approved,2,0,2,2
canceled,625,141,550,619
created,5,5,5,5
delivered,96478,14,2,8
invoiced,314,0,314,314
processing,301,0,301,301
shipped,1107,0,0,1107
unavailable,609,0,609,609


In [45]:
is_delivered = orders_clean['order_status'].eq('delivered')

orders_clean['flag_delivered_missing_approval'] = (
    is_delivered & orders_clean['order_approved_at'].isna()
)

orders_clean['flag_delivered_missing_carrier'] = (
    is_delivered & orders_clean['order_delivered_carrier_date'].isna()
)

orders_clean['flag_canceled_has_delivery_date'] = (
    orders_clean['order_status'].eq('canceled')
    & orders_clean['order_delivered_customer_date'].notna()
)

In [46]:
flag_cols = [
    'flag_delivered_missing_date',
    'flag_delivered_missing_approval',
    'flag_delivered_missing_carrier',
    'flag_carrier_before_purchase',
    'flag_delivery_before_carrier',
    'flag_canceled_has_delivery_date'
]

# 每一种问题分别触发了多少笔订单
print(orders_clean[flag_cols].sum())

# 同一订单只要触发任意一项，就进入待核查清单
orders_clean['flag_needs_review'] = (
    orders_clean[flag_cols].any(axis=1)
)

print('待核查订单总数：',
      orders_clean['flag_needs_review'].sum())

flag_delivered_missing_date          8
flag_delivered_missing_approval     14
flag_delivered_missing_carrier       2
flag_carrier_before_purchase       166
flag_delivery_before_carrier        23
flag_canceled_has_delivery_date      6
dtype: int64
待核查订单总数： 218
